A new corpus and a new gold reward


In [1]:

!pip install -q gradio pandas matplotlib

In [ ]:

# ============================================================
# PROJECT: A NEW CORPUS AND A NEW GOLD REWARD
# Domain: Weather Reports
# Platform: Google Colab
# Interface: Gradio
# ============================================================

import random
import re
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from collections import defaultdict
from datetime import datetime

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_NAME = "Weather Reward AI"
PROJECT_VERSION = "1.0"

DEFAULT_CORPUS = [
    "rain is likely in the north <eos>",
    "sunny and warm with 28 degrees <eos>",
    "cloudy with light wind in the north <eos>",
    "sunny in the south with 31 degrees <eos>",
    "light rain and 19 degrees in the south <eos>"
]

# ============================================================
# 2. CORPUS PROCESSING
# ============================================================

def clean_sentence(sentence):
    """Remove leading/trailing spaces and normalize EOS."""
    sentence = str(sentence).strip()

    if not sentence:
        return ""

    words = sentence.split()

    # Remove repeated end tokens.
    while words and words[-1] == "<eos>":
        words.pop()

    if not words:
        return ""

    return " ".join(words) + " <eos>"


def parse_corpus(corpus_text):
    """Convert a multiline textbox into a valid corpus."""
    if not corpus_text or not corpus_text.strip():
        return DEFAULT_CORPUS.copy()

    sentences = []

    for line in corpus_text.splitlines():
        sentence = clean_sentence(line)

        if sentence:
            sentences.append(sentence)

    return sentences if sentences else DEFAULT_CORPUS.copy()


def get_report_words(sentence):
    """Return sentence words without special tokens."""
    return [
        word
        for word in sentence.split()
        if word not in ("<eos>", "<bos>")
    ]


# ============================================================
# 3. GOLD REWARD FUNCTION
# ============================================================

def gold_reward(sentence):
    """
    Gold reward:
    +1.0 if a numeric temperature is present.
    -0.1 for each word.

    Formula:
    R(s) = I(temperature in s) - 0.1 * length(s)
    """
    words = get_report_words(sentence)

    has_temperature = any(
        word.isdigit() for word in words
    )

    temperature_reward = 1.0 if has_temperature else 0.0
    length_penalty = 0.1 * len(words)

    reward = temperature_reward - length_penalty

    return round(reward, 4)


def reward_details(sentence):
    """Explain how the reward was calculated."""
    words = get_report_words(sentence)

    has_temperature = any(
        word.isdigit() for word in words
    )

    temperature_reward = 1.0 if has_temperature else 0.0
    length_penalty = 0.1 * len(words)
    final_reward = temperature_reward - length_penalty

    details = (
        f"Sentence: {sentence}\n\n"
        f"Word count: {len(words)}\n"
        f"Numeric temperature found: "
        f"{'Yes' if has_temperature else 'No'}\n"
        f"Temperature reward: {temperature_reward:+.2f}\n"
        f"Length penalty: {-length_penalty:+.2f}\n"
        f"Final gold reward: {final_reward:+.2f}"
    )

    return details


# ============================================================
# 4. REFERENCE MODEL
# ============================================================

def build_reference_model(corpus):
    """
    Build a word-transition model.
    Each word stores the words observed after it.
    """
    follow = defaultdict(list)
    valid_sentences = []

    for sentence in corpus:
        sentence = clean_sentence(sentence)

        if not sentence:
            continue

        valid_sentences.append(sentence)
        words = ["<bos>"] + sentence.split()

        for current_word, next_word in zip(
            words, words[1:]
        ):
            follow[current_word].append(next_word)

    return follow, valid_sentences


# ============================================================
# 5. SENTENCE GENERATION
# ============================================================

def generate_sentence(
    follow,
    rng,
    max_words=12
):
    """
    Generate a report using observed word transitions.
    """
    current_word = "<bos>"
    output = []

    for _ in range(max_words):
        choices = follow.get(current_word, [])

        if not choices:
            break

        current_word = rng.choice(choices)
        output.append(current_word)

        if current_word == "<eos>":
            break

    return " ".join(output)


def generate_reports(
    corpus_text,
    number_of_reports,
    random_seed,
    max_words
):
    """Generate reports and calculate their gold rewards."""
    corpus = parse_corpus(corpus_text)
    follow, corpus = build_reference_model(corpus)

    rng = random.Random(int(random_seed))

    rows = []

    for report_number in range(
        1, int(number_of_reports) + 1
    ):
        sentence = generate_sentence(
            follow,
            rng,
            int(max_words)
        )

        score = gold_reward(sentence)
        words = get_report_words(sentence)

        has_temperature = any(
            word.isdigit() for word in words
        )

        rows.append({
            "Report ID": report_number,
            "Generated Report": sentence,
            "Word Count": len(words),
            "Temperature": (
                "Yes" if has_temperature else "No"
            ),
            "Gold Reward": score
        })

    results = pd.DataFrame(rows)

    average_reward = results["Gold Reward"].mean()
    best_index = results["Gold Reward"].idxmax()

    best_report = results.loc[
        best_index, "Generated Report"
    ]

    summary = (
        "### Generation Summary\n\n"
        f"- Reports generated: {len(results)}\n"
        f"- Average reward: {average_reward:.2f}\n"
        f"- Best reward: {results['Gold Reward'].max():.2f}\n"
        f"- Lowest reward: {results['Gold Reward'].min():.2f}\n"
        f"- Reports containing numeric temperatures: "
        f"{(results['Temperature'] == 'Yes').sum()}\n"
        f"- Highest-scoring report: {best_report}"
    )

    # Plot reward distribution.
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.hist(
        results["Gold Reward"],
        bins=min(10, max(3, len(results))),
        edgecolor="black"
    )

    ax.set_title("Gold Reward Distribution")
    ax.set_xlabel("Gold Reward")
    ax.set_ylabel("Number of Reports")
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()

    return results, summary, fig


# ============================================================
# 6. CORPUS ANALYSIS
# ============================================================

def analyze_corpus(corpus_text):
    """Show corpus statistics and transition information."""
    corpus = parse_corpus(corpus_text)
    follow, corpus = build_reference_model(corpus)

    rows = []

    for index, sentence in enumerate(corpus, start=1):
        words = get_report_words(sentence)

        has_temperature = any(
            word.isdigit() for word in words
        )

        rows.append({
            "Sentence ID": index,
            "Sentence": sentence,
            "Word Count": len(words),
            "Temperature": (
                "Yes" if has_temperature else "No"
            ),
            "Gold Reward": gold_reward(sentence)
        })

    dataframe = pd.DataFrame(rows)

    total_words = sum(
        len(get_report_words(sentence))
        for sentence in corpus
    )

    vocabulary = set()

    for sentence in corpus:
        vocabulary.update(sentence.split())

    temperature_count = sum(
        1 for sentence in corpus
        if any(
            word.isdigit()
            for word in get_report_words(sentence)
        )
    )

    report = (
        "### Corpus Statistics\n\n"
        f"- Total sentences: {len(corpus)}\n"
        f"- Total words: {total_words}\n"
        f"- Unique tokens, including EOS: {len(vocabulary)}\n"
        f"- Unique transition keys: {len(follow)}\n"
        f"- Reports with numeric temperatures: {temperature_count}\n"
        f"- Average sentence length: "
        f"{total_words / max(len(corpus), 1):.2f} words\n"
        f"- Temperature coverage: "
        f"{100 * temperature_count / max(len(corpus), 1):.1f}%"
    )

    return dataframe, report


# ============================================================
# 7. MANUAL REWARD CALCULATOR
# ============================================================

def calculate_manual_reward(sentence):
    """Evaluate a user-entered weather report."""
    if not sentence or not sentence.strip():
        return (
            "Please enter a weather report.",
            0.0
        )

    score = gold_reward(sentence)
    details = reward_details(sentence)

    return details, score


# ============================================================
# 8. TRANSITION EXPLORER
# ============================================================

def explore_transitions(corpus_text):
    """Display words and their observed next-word options."""
    corpus = parse_corpus(corpus_text)
    follow, corpus = build_reference_model(corpus)

    rows = []

    for word in sorted(follow.keys()):
        next_words = follow[word]

        rows.append({
            "Current Word": word,
            "Possible Next Words": ", ".join(next_words),
            "Transition Count": len(next_words)
        })

    dataframe = pd.DataFrame(rows)

    explanation = (
        "### How the Reference Model Works\n\n"
        "1. Each sentence begins with `<bos>`.\n"
        "2. Words are read from left to right.\n"
        "3. Every observed next-word transition is stored.\n"
        "4. Generation randomly selects an observed next word.\n"
        "5. Generation stops at `<eos>` or the word limit.\n\n"
        "This is a simple transition model, not a neural network."
    )

    return dataframe, explanation


# ============================================================
# 9. COMPARISON OF GOLD REWARDS
# ============================================================

def compare_examples():
    """Prepare illustrative examples of the reward formula."""
    examples = [
        "sunny 28",
        "sunny with 28 degrees",
        "sunny and warm with 28 degrees",
        "rain is likely in the north",
        "cloudy with light wind in the north",
        "rain"
    ]

    rows = []

    for sentence in examples:
        words = get_report_words(sentence)
        has_temperature = any(
            word.isdigit() for word in words
        )

        rows.append({
            "Weather Report": sentence,
            "Word Count": len(words),
            "Has Temperature": (
                "Yes" if has_temperature else "No"
            ),
            "Gold Reward": gold_reward(sentence)
        })

    dataframe = pd.DataFrame(rows)

    dataframe = dataframe.sort_values(
        "Gold Reward",
        ascending=False
    ).reset_index(drop=True)

    return dataframe


# ============================================================
# 10. GRADIO WEB APPLICATION
# ============================================================

CUSTOM_CSS = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=CUSTOM_CSS,
    title=PROJECT_NAME
) as demo:

    gr.Markdown(
        """
        # 🌦️ Weather Reward AI
        ### New Corpus and a New Gold Reward

        An interactive demonstration of a weather-domain
        corpus, a reference language model, and a gold reward
        function that prefers short reports stating a temperature.
        """
    )

    with gr.Row():
        gr.Markdown(
            """
            **Domain:** Weather Reports

            **Language:** Python

            **Model:** Word-transition reference model

            **Reward:** Temperature bonus minus length penalty
            """
        )

    gr.Markdown("---")

    # Shared corpus editor
    gr.Markdown("## 📚 Weather Corpus")

    corpus_input = gr.Textbox(
        label="Corpus: one sentence per line",
        value="\n".join(
            sentence.replace(" <eos>", "")
            for sentence in DEFAULT_CORPUS
        ),
        lines=7,
        placeholder="Enter one weather report per line"
    )

    gr.Markdown(
        "You can edit the corpus above. "
        "The model and its statistics use the edited corpus."
    )

    with gr.Tab("Dashboard"):
        gr.Markdown("## Project Overview")

        gr.Markdown(
            """
            This application demonstrates four main tasks:

            1. Create and inspect a domain-specific corpus.
            2. Build a reference model from observed word transitions.
            3. Generate new sentences using random sampling.
            4. Evaluate generated sentences with a gold reward function.
            """
        )

        dashboard_button = gr.Button(
            "Analyze Current Corpus",
            variant="primary"
        )

        dashboard_table = gr.Dataframe(
            label="Corpus Analysis",
            interactive=False
        )

        dashboard_stats = gr.Markdown()

        dashboard_button.click(
            fn=analyze_corpus,
            inputs=[corpus_input],
            outputs=[dashboard_table, dashboard_stats]
        )

    with gr.Tab("Generate Reports"):
        gr.Markdown("## 🤖 Reference Model Generator")

        with gr.Row():
            report_count = gr.Slider(
                minimum=1,
                maximum=50,
                value=10,
                step=1,
                label="Number of reports"
            )

            seed_input = gr.Number(
                value=0,
                precision=0,
                label="Random seed"
            )

            word_limit = gr.Slider(
                minimum=3,
                maximum=30,
                value=12,
                step=1,
                label="Maximum generated tokens"
            )

        generate_button = gr.Button(
            "Generate Weather Reports",
            variant="primary"
        )

        generated_table = gr.Dataframe(
            label="Generated Weather Reports",
            interactive=False
        )

        generation_summary = gr.Markdown()

        reward_plot = gr.Plot(
            label="Reward Distribution"
        )

        generate_button.click(
            fn=generate_reports,
            inputs=[
                corpus_input,
                report_count,
                seed_input,
                word_limit
            ],
            outputs=[
                generated_table,
                generation_summary,
                reward_plot
            ]
        )

    with gr.Tab("Gold Reward Calculator"):
        gr.Markdown("## 🧮 Manual Reward Calculator")

        manual_sentence = gr.Textbox(
            label="Enter a weather report",
            value="sunny with 28 degrees",
            lines=3,
            placeholder="Example: sunny with 28 degrees"
        )

        calculate_button = gr.Button(
            "Calculate Gold Reward",
            variant="primary"
        )

        reward_details_output = gr.Textbox(
            label="Reward Calculation Details",
            lines=8,
            interactive=False
        )

        reward_value_output = gr.Number(
            label="Final Gold Reward",
            interactive=False
        )

        calculate_button.click(
            fn=calculate_manual_reward,
            inputs=[manual_sentence],
            outputs=[
                reward_details_output,
                reward_value_output
            ]
        )

    with gr.Tab("Transition Explorer"):
        gr.Markdown("## 🔎 Word Transition Explorer")

        transition_button = gr.Button(
            "Explore Word Transitions",
            variant="primary"
        )

        transition_table = gr.Dataframe(
            label="Reference Model Transitions",
            interactive=False
        )

        transition_explanation = gr.Markdown()

        transition_button.click(
            fn=explore_transitions,
            inputs=[corpus_input],
            outputs=[
                transition_table,
                transition_explanation
            ]
        )

    with gr.Tab("Reward Comparison"):
        gr.Markdown("## 📊 Compare Reward Scores")

        gr.Markdown(
            "These examples illustrate how temperature detection "
            "and report length affect the reward."
        )

        comparison_button = gr.Button(
            "Compare Example Reports",
            variant="primary"
        )

        comparison_table = gr.Dataframe(
            label="Reward Comparison",
            interactive=False
        )

        comparison_button.click(
            fn=compare_examples,
            inputs=[],
            outputs=[comparison_table]
        )

    with gr.Tab("Project Documentation"):
        gr.Markdown(
            """
            ## Project Objective

            Replace a previous corpus with weather reports and define
            a gold reward function that prefers short reports containing
            a numeric temperature.

            ## Gold Reward Formula

            `R(s) = I(temperature in s) - 0.1 * L(s)`

            Here, `I` equals 1 when a numeric token exists and 0 otherwise.
            `L(s)` is the number of words, excluding special tokens.

            ## Reference Model

            The reference model stores the words that follow each word
            in the training corpus. It samples from those observed
            transitions to generate sentences.

            ## Important Limitation

            The reference model is a simple word-transition model.
            It does not learn complex grammar or meteorological facts.
            The gold reward is a scoring rule, not a learned reward model.

            ## Expected Result

            The application allows users to inspect the corpus, generate
            weather reports, evaluate rewards, compare examples, and
            explore the word-transition model.
            """
        )

    gr.Markdown("---")

    gr.Markdown(
        "Weather Reward AI | Python • Pandas • Matplotlib • Gradio"
    )


# ============================================================
# 11. LAUNCH APPLICATION
# ============================================================

print("Starting Weather Reward AI...")
print("Project version:", PROJECT_VERSION)
print("Corpus sentences:", len(DEFAULT_CORPUS))
print("Launch time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

demo.launch(
    share=True,
    debug=True
)


/tmp/ipykernel_729/2935647360.py:440: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Starting Weather Reward AI...
Project version: 1.0
Corpus sentences: 5
Launch time: 2026-10-13 15:32:16
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4ede89c5bd36861b16.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
